# 02 — Feature Engineering

This notebook explains the 29 features we build for every trip prediction,
and the reasoning behind each design decision.

In [ ]:
import sys
sys.path.append('..')

import math
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from preprocessing import build_features, compute_delay_min, stable_hash

## 1. Why Sin/Cos for Time Features?

If we encode hour as a plain integer (0–23), the model sees hour 23 and hour 0 as far apart — but midnight is continuous. A train at 23:55 and one at 00:05 are 10 minutes apart, not 23 hours apart.

**Solution:** encode each cyclic variable as two values — sin and cos — so the distance between any two times reflects their actual closeness on the cycle.

In [ ]:
hours = np.arange(24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw integer encoding — 23 and 0 look far apart
axes[0].plot(hours, hours, 'o-', color='tomato')
axes[0].set_title('Raw hour encoding (broken at midnight)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Feature value')
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(23, color='gray', linestyle='--', alpha=0.5)

# Cyclical encoding — 23 and 0 are close on the circle
axes[1].scatter(hour_sin, hour_cos, c=hours, cmap='hsv', s=80, zorder=3)
theta = np.linspace(0, 2*np.pi, 200)
axes[1].plot(np.sin(theta), np.cos(theta), 'gray', alpha=0.3)
for h in [0, 6, 12, 18, 23]:
    axes[1].annotate(f'{h}h', (hour_sin[h], hour_cos[h]), fontsize=9,
                     xytext=(5, 5), textcoords='offset points')
axes[1].set_title('Cyclical sin/cos encoding (midnight is continuous)')
axes[1].set_xlabel('sin(hour)')
axes[1].set_ylabel('cos(hour)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()
print('Hour 23 sin/cos:', round(hour_sin[23], 3), round(hour_cos[23], 3))
print('Hour  0 sin/cos:', round(hour_sin[0],  3), round(hour_cos[0],  3))
print('Euclidean distance between hour 23 and 0:', 
      round(math.sqrt((hour_sin[23]-hour_sin[0])**2 + (hour_cos[23]-hour_cos[0])**2), 3),
      '← small, as expected')

We apply this to: **hour, minute, weekday, calendar week, month** — all five have natural cycles.

## 2. Categorical Features via Stable Hashing

Categorical strings like `line='S1'` or `direction='Plochingen'` can't be fed directly to a decision tree regressor. We hash them to stable integers.

**Why MD5-based hashing instead of label encoding?**
- Label encoding needs to know all categories upfront — we don't (new lines can appear)
- MD5 is deterministic across runs and machines
- We modulo the hash to keep values bounded (e.g., `% 1000` for lines)

In [ ]:
lines = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'RB', 'RE']
print('Line → hash (mod 1000):')
for line in lines:
    print(f'  {line:4s} → {stable_hash(line, 1000)}')

print('\nSame input always gives same hash:')
print('S1 hashed twice:', stable_hash('S1', 1000), stable_hash('S1', 1000))

## 3. Real-Time Signal: Departure Delay

The most powerful single feature is **how late has this train already departed?**
A train that departed 8 minutes late will almost certainly arrive late too.

We use two features:
- `dep_delay_min` — the departure delay in minutes (capped to ±120)
- `dep_delay_known` — 1 if we have a real departure time, 0 if the train hasn't departed yet

The `_known` flag lets the model learn to discount the feature when it's unavailable.

In [ ]:
# Example: train scheduled dep 08:00, actually departed 08:07
from datetime import datetime

example_row = {
    'scheduled_dep': datetime(2026, 5, 30, 8, 0),
    'actual_dep':    datetime(2026, 5, 30, 8, 7),   # 7 min late
    'scheduled_arr': datetime(2026, 5, 30, 8, 25),
    'actual_arr':    None,   # not arrived yet — prediction time
    'train_type': 'S',
    'line': 'S1',
    'station_name': 'Stuttgart Hbf',
    'station_eva': '8000096',
    'direction': 'Plochingen',
    'cancelled': False,
}

features = build_features(example_row)
print(f"dep_delay_min   = {features['dep_delay_min']:+.1f} min")
print(f"dep_delay_known = {features['dep_delay_known']}")
print(f"hour            = {features['hour']}")
print(f"hour_sin        = {features['hour_sin']:.4f}")
print(f"hour_cos        = {features['hour_cos']:.4f}")

## 4. Route Topology Features

The DB API's `ppth` field gives us the full planned path: a `|`-separated list of all stops.
We extract three features from it:

| Feature | What it captures |
|---|---|
| `route_length` | Total stops on this route — longer routes accumulate more delay |
| `route_position_pct` | Where is the current station in the route (0=start, 1=end) |
| `route_hash` | Which specific route — the model learns per-route delay patterns |

In [ ]:
# S1 from Kirchheim to Plochingen passes through Stuttgart Hbf
example_ppth = 'Kirchheim (T)|Denkendorf|Plochingen|Esslingen|Bad Cannstatt|Stuttgart Hbf|Zuffenhausen'

example_row['ppth'] = example_ppth
features = build_features(example_row)

stops = example_ppth.split('|')
print(f'Route: {example_ppth}')
print(f'Stops: {stops}')
print()
print(f"route_length         = {features['route_length']}")
print(f"route_position       = {features['route_position']}  ← Stuttgart Hbf is stop #{features['route_position']}")
print(f"route_position_pct   = {features['route_position_pct']:.2f}  ← {features['route_position_pct']*100:.0f}% through the route")
print(f"route_position_known = {features['route_position_known']}")
print(f"route_hash           = {features['route_hash']}")

## 5. Upstream Delay Propagation

If we've already seen this train at a different station earlier in the scrape cycle (or a previous cycle),
we know its last confirmed delay. This is the most direct real-time signal for delay propagation.

In `pipeline.py`:
```python
train_delays: dict = {}  # {train_id → last confirmed delay}

# Before prediction:
row['upstream_delay_min'] = train_delays.get(row['train_id'])

# After learning:
if delay is not None:
    train_delays[row['train_id']] = delay
```

In [ ]:
# Simulate: train was 5 min late at Bad Cannstatt, now approaching Stuttgart Hbf
example_row['upstream_delay_min'] = 5.0
features = build_features(example_row)

print(f"upstream_delay_min   = {features['upstream_delay_min']:+.1f} min")
print(f"upstream_delay_known = {features['upstream_delay_known']}")
print()
print('Without upstream info:')
example_row['upstream_delay_min'] = None
f2 = build_features(example_row)
print(f"  upstream_delay_min   = {f2['upstream_delay_min']:+.1f}")
print(f"  upstream_delay_known = {f2['upstream_delay_known']}  ← model knows to discount this")

## 6. Complete Feature List

All 29 features in the final vector:

In [ ]:
example_row['upstream_delay_min'] = 5.0
features = build_features(example_row)

print(f'Total features: {len(features)}')
print()
groups = [
    ('Time (raw)',         ['hour', 'weekday', 'minutes_since_midnight']),
    ('Time (cyclical)',    ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
                            'weekday_sin', 'weekday_cos', 'week_sin', 'week_cos',
                            'month_sin', 'month_cos']),
    ('Train identity',    ['station_eva', 'train_type', 'line', 'line_missing',
                            'direction', 'cancelled', 'arr_missing']),
    ('Real-time signal',  ['dep_delay_min', 'dep_delay_known']),
    ('Route topology',    ['route_length', 'route_hash', 'route_position',
                            'route_position_known', 'route_position_pct']),
    ('Upstream delay',    ['upstream_delay_min', 'upstream_delay_known']),
]
for group_name, keys in groups:
    print(f'  [{group_name}]')
    for k in keys:
        v = features[k]
        print(f'    {k:28s} = {v}')
    print()